# SDA-DSC-212 — Time Series Analysis and Forecasting
## Lab 6 — The Rolling-Origin Backtest Report

**Duration** 50 minutes · **Work in pairs** · `git checkout lab6-start`

> The one rule for this whole course: every feature, every split and every scaler
> must be computable using only data that existed at the forecast origin.

### Tasks

| min | task |
|---|---|
| 5 | Find the **fit-once leak** in the inherited backtest. Show that its MASE is absurd. Record it in `BACKTEST.md`. **Do not fix it yet.** |
| 12 | Configure a proper rolling origin: expanding window, one-year initial window, step 24 h, horizon 24 h, training data ending **at** each origin. At least 50 origins, at least three model families. |
| 10 | Aggregate the per-origin errors: the **mean and the spread** of MAE, MAPE and MASE per model, every model reported relative to the seasonal-naive baseline. |
| 8 | Probabilistic scoring for the interval model: mean pinball loss and empirical coverage across origins. |
| 10 | `diebold_mariano` on the top two point models. Record the significance. Decide the champion and **state the rule you applied**. |
| 5 | Write `BACKTEST.md`: the table, the DM verdict, the champion, and the conditions under which the champion loses. Commit. |

> This is the headline deliverable of the course. Everything before it produced a model;
> this lab produces the **evidence** that lets somebody decide whether to run it.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

DATA = "../data"

In [ ]:
from tayyar.data.load import load_demand, load_calendar, join_calendar
from tayyar.features.build import build_features

df = join_calendar(load_demand(f"{DATA}/ksa_grid_demand.csv"),
                   load_calendar(f"{DATA}/ksa_calendar.csv"))
feat = build_features(df[["demand_mw", "temp_c", "is_weekend",
                          "is_ramadan", "is_eid", "is_national_day"]])
Xcols = [c for c in feat.columns if c != "demand_mw"]
X, yy = feat[Xcols].astype(float), feat["demand_mw"]
y = df["demand_mw"].astype(float)

CUT_TRAIN, CUT_CAL = "2023-09-30 23:00", "2023-10-31 23:00"
print(f"{len(feat):,} hourly rows x {len(Xcols)} features "
      f"({df.index.min():%Y-%m-%d} -> {df.index.max():%Y-%m-%d})")

### Task 1 — the backtest you inherited

Run the next cell. It loops over sixty origins, it forecasts twenty-four hours at each, and
it reports a MASE of 0.017 — the model makes under two per cent of the error of the free
seasonal-naive baseline.

Nobody in this industry has a day-ahead load model with a MASE of 0.017.

Five minutes: find the line, and write it into `BACKTEST.md`. Do not fix it yet.

In [ ]:
# ---------------------------------------------------------------------------
# The backtest you inherited
#
# The note attached to it read:
#   "60-origin rolling backtest, day-ahead. MASE 0.02. Ready for the report."
#
# It runs. It loops over sixty origins. It prints a MASE. One line in it makes
# every number it produces meaningless. Find that line before you write any code.
# ---------------------------------------------------------------------------
import lightgbm as lgb
from tayyar.eval.metrics import all_metrics

ok = X.notna().all(axis=1)
Xall, yall = X[ok], yy[ok]

# "train the model"
starter = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.08, num_leaves=255,
                            min_child_samples=2, verbose=-1,
                            random_state=7).fit(Xall, yall)

# "roll the origin forward one day at a time and score the next 24 hours"
ORIGINS_STARTER = pd.date_range("2023-11-01 23:00", periods=60, freq="D",
                               tz="Asia/Riyadh")
rows = []
for o in ORIGINS_STARTER:
    pos = X.index.get_loc(o)
    block = X.iloc[pos + 1: pos + 25]              # the 24 hours being "forecast"
    actual = yy.iloc[pos + 1: pos + 25]
    hist = yy.iloc[pos - 24 * 365: pos]            # in-sample scale for MASE
    rows.append(all_metrics(actual, starter.predict(block), hist, 24))

starter_bt = pd.DataFrame(rows)
print(f"origins scored : {len(starter_bt)}")
print(starter_bt[["MAE", "RMSE", "MAPE_%", "MASE", "bias"]].mean().round(4).to_string())

In [ ]:
# TODO (5 min) — do not change the cell above yet.
#
# The MASE is 0.017. Before you look for a bug, look for the reason it CANNOT be real:
# what does MASE = 0.017 claim, in words, about this model versus copying yesterday?
#
#   1. Collect every hour the loop scores:
#        X.index[pos+1 : pos+25] for every origin
#      How many are there? How many of them are in the index the model was fitted on?
#
#   2. Print the last hour of the training data and the last hour the backtest claims
#      to forecast. Which is later?
#
#   3. Refit the SAME model on data ending at CUT_TRAIN, score the same sixty origins,
#      and put the two MASE numbers side by side.
#
# Write it up in BACKTEST.md: the line, the mechanism, the two numbers. Do not fix it.
#
# HINT: a backtest is a claim about what you knew at a moment in time. Read the cell
#       above in that light and ask, at the line that fits the model, what time it is.

### Task 2 — a rolling origin that is actually rolling

**Why one split is not evidence.** A single train/test split estimates skill from *one*
realisation of the future. It gives you a number with no standard error attached, and the
luck of which weeks landed in the test window is baked into it — a mild fortnight flatters
every model, a fortnight with two calendar shocks flatters the one that happens to lag.
Change the split date by two weeks and rankings flip. A backtest replaces that single number
with a **distribution of skill over many origins**: the mean tells you what to expect, the
spread tells you what to plan for, and the per-origin series tells you *when* you are exposed.

**Expanding vs sliding.**

| window | train set at origin t | use it when |
|---|---|---|
| **expanding** | everything from the start of history to t; grows at every origin | the regime is stable and more history is more signal |
| **sliding** | a fixed-length block ending at t; forgets the oldest data | the regime drifts — a new industrial load, a tariff change, a post-refurbishment plant |

Expanding is the default here: three years of Saudi grid demand is one regime plus growth.
A sliding window is the right answer the moment you can name the date the world changed.

**"Refit at each origin" is two different requirements, and only one is negotiable.**

1. **Correctness — not negotiable.** The training data must end **at or before the origin**.
   Nothing the model saw may come from after the hour you are pretending to stand at. Break
   this and you have the Task 1 leak.
2. **Cadence — a budget decision you must state.** How often you re-estimate between
   origins. Refitting the 24-model direct forecaster at all 60 origins is 1,440 LightGBM
   fits, roughly an hour of CPU on a laptop, and three families make it three hours.

So we do both, honestly. The 60-origin table below uses models estimated **once at the
30 September boundary** — a monthly retrain cadence, with training data ending a full month
before the first origin and never after any origin — and then we **measure the cadence
choice** by refitting at every origin for six of the sixty and comparing. Say which of the
two you ran, every time you publish a backtest. Most published "rolling backtests" are the
periodic-retrain kind and do not admit it.

In [ ]:
from tayyar.eval.backtest import BacktestConfig, origins
from tayyar.models.lgbm_forecaster import DirectLGBMForecaster
from tayyar.models.quantile import QuantileLGBM

# TODO (12 min)
#
#   1. cfg = BacktestConfig(horizon=24, n_origins=60, step=24,
#                           initial_train=24*365, window="expanding", m=24)
#      folds = list(origins(y.index, cfg))
#      ORIGINS = [y.index[tr.stop - 1] for tr, te in folds]
#      Print len(folds) — the report needs at least 50 — and the first/last origin.
#      Print fold 0 and fold 59 for window="expanding" and again for "sliding".
#      Which one moves its train START position, and what does that mean?
#
#   2. Fit the candidates on data ending at CUT_TRAIN (30 September), a month before
#      the first origin:
#        lgbm = DirectLGBMForecaster(horizon=24).fit(Xtr, ytr)          # ~1 min
#        qm   = QuantileLGBM((0.05, 0.50, 0.95), horizon=24).fit(...)   # ~3 min
#      Start them, then read the next markdown cell while they run.
#
#   3. backtest(predict_fn, name) -> one metrics row per origin. At each origin:
#        Xo     = X.iloc[[pos]]                  ONE feature row — the origin's
#        actual = yy.iloc[pos+1 : pos+25]
#        hist   = yy.iloc[pos - cfg.initial_train : pos]     the MASE scale
#      Run it for all three families: LightGBM direct, quantile q50, seasonal-naive.
#      Keep the raw (actual, pred) arrays per origin — Task 5 needs them.
#
#   4. Then measure the CADENCE choice: for six origins, refit the direct model with
#      training data ending AT the origin and compare its MASE with the fixed fit.
#      Time it, and report what sixty refits would have cost.
#
# HINT: the forecast for hour t+7 must come from the feature row at the ORIGIN, not
#       from the feature row at t+7 — that row contains the lag-1 demand of t+6, which
#       nobody has at 23:00. If your MAPE comes out near 1%, that is what happened.

### Task 3 — the comparison table, and the failure mode of every column in it

| metric | what it is | what it is good for | how it fails |
|---|---|---|---|
| **MAE** | mean absolute error | same units as the series; robust to outliers; the number an engineer can act on | not comparable across series of different scale |
| **RMSE** | root mean squared error | penalises large misses, which is right when the cost is convex | dominated by a handful of outliers; a single bad hour rewrites the ranking |
| **MAPE** | mean absolute percentage error | scale-free and immediately intuitive | explodes as the denominator approaches zero, and is asymmetric — an over-forecast is capped at 100% while an under-forecast is unbounded, so it quietly prefers models that under-predict |
| **sMAPE** | symmetric MAPE | bounded, so it cannot explode to 4,000% | still unstable near zero, and "symmetric" is a misnomer — it is not symmetric in the errors |
| **MASE** | MAE / MAE of the **in-sample seasonal-naive** | scale-free, symmetric, defined at zero, comparable across series | needs a sensible seasonal period `m`, and the denominator is a modelling choice you must state |

**MASE < 1 means you beat the free baseline.** MASE > 1 means you should have shipped the
baseline and gone home. It is the only column in this table that is simultaneously scale-free
and safe near zero, which is why it is the column the champion is chosen on.

Report the **spread**, not only the mean. The mean is what the model does on an average day;
the maximum is what the control room lives through on the worst one.

In [ ]:
# TODO (10 min)
#
#   1. summary = bt.groupby("model")[MET].mean().round(3).sort_values("MASE")
#      with MET = ["MAE","RMSE","MAPE_%","WAPE_%","MASE","bias"]
#   2. The SPREAD is not optional: .agg(["mean","std","min","max"]) on MAE, MAPE_%, MASE.
#   3. Report every model RELATIVE to the seasonal-naive baseline:
#        - MAE as a percentage of the baseline's MAE
#        - how many of the 60 origins the model was worse than the baseline on
#   4. Plot per-origin MASE for all three families, with a line at MASE = 1, and a
#      boxplot of the three distributions beside it.
#
#   Then answer in BACKTEST.md, in one sentence each:
#     - the seasonal-naive baseline scores MASE 0.77, not 1.00. Why?
#     - what is the champion's WORST origin, and what would that day have cost?
#
# HINT: MASE divides by the in-sample seasonal-naive MAE over the preceding YEAR.
#       Which season dominates that year, and which season are these origins in?

### Task 4 — scoring the interval, not just the point

A point forecast is scored by how far it lands from the truth. A **quantile** forecast has to
be scored by something that knows which quantile it claims to be — that is pinball loss,
asymmetric on purpose: at τ = 0.95 an under-prediction is penalised nineteen times more than
an over-prediction, and its minimiser is the 95th percentile.

Two numbers, and you need both:

- **mean pinball loss** — sharpness and calibration together, averaged over the quantile grid.
  Lower is better; it is a proper score, so it cannot be gamed by widening the band.
- **empirical coverage** — the fraction of hours the interval actually contained the truth.
  Coverage is the constraint; pinball is the objective. An interval that covers 100% by being
  20,000 MW wide scores badly on pinball and is useless in a control room.

In [ ]:
from tayyar.eval.pinball import pinball_loss, coverage
from tayyar.models.conformal import CQR

# TODO (8 min)
#
#   1. Predict all three quantiles at each of the 60 origins:
#        P[tau] -> array (n_origins, 24);  Y -> the matching actuals
#   2. Calibrate CQR on OCTOBER (X[CUT_TRAIN:CUT_CAL]) — held out and adjacent in
#      regime, disjoint from every origin — and apply q-hat to the band.
#   3. Report coverage and mean width, raw and calibrated, against the nominal 0.90.
#   4. pinball_loss at each tau and the MEAN over the grid.
#   5. Then the same two quantities PER ORIGIN: mean, sd, min, max, and how many
#      origins covered less than 0.80. Plot both series against the origin date.
#
# HINT: the marginal coverage will land on 0.90 and look finished. Count the origins
#       below 0.80 before you write that number into the report — an average over
#       days is not a promise about any day.

### Task 5 — Diebold-Mariano: is the gap real?

The two LightGBM variants differ by about 16 MW of MAE on a 30,000 MW system. Before anybody
ships the marginally better one, answer the only question that matters: **could that gap be
noise?**

Diebold-Mariano tests **H0: the two forecasts have equal expected loss**, using the
*loss-differential series* `d_t = |e1_t| - |e2_t|`. It is a test on a series, so it accounts
for the autocorrelation the 24-hour horizon induces (Newey-West, with the Harvey-Leybourne-
Newbold small-sample correction). A significant negative statistic says model 1 is genuinely
better; a non-significant one says **you cannot distinguish them, so ship the simpler or
cheaper model**.

That last sentence is the promotion rule. Statistical significance **gates** added complexity:
a sub-1% "win" that fails DM does not justify a second model in production, a second thing to
monitor, and a second thing to wake somebody at 02:00.

In [ ]:
from tayyar.eval.diebold_mariano import diebold_mariano

# TODO (10 min)
#
#   1. Concatenate the per-origin actuals and predictions into 1,440-hour vectors
#      (they must be PAIRED — same origins, same hours, for every model).
#   2. diebold_mariano(actual, f1, f2, h=cfg.horizon) for
#        LightGBM (direct)  vs  Seasonal-naive
#        LightGBM (direct)  vs  LightGBM (quantile q50)
#      Report dm, p, n and the mean loss differential for both.
#   3. Plot the mean loss differential PER ORIGIN for each comparison. What does a
#      non-significant result look like on that plot?
#   4. Name the champion, and WRITE DOWN THE RULE you applied. A rule that only names
#      the winner is not a rule.
#
# HINT: h=24 matters. The loss differentials of a 24-hour-ahead forecast are
#       autocorrelated; the test uses a Newey-West long-run variance with h-1 lags,
#       and passing h=1 would understate the variance and manufacture significance.

### When does the champion lose?

The mean is what you report; the per-origin breakdown is what tells you **which weeks** the
champion fails on, and that is the section of the report an operator actually reads. Two
questions, and the second matters more:

1. Which origins in this window went worst, and what was happening on them?
2. **Which regimes does this window contain no evidence about at all?**

In [ ]:
# TODO (part of Tasks 5-6)
#
#   1. Sort the champion's per-origin table by MASE and take the worst six. For each,
#      print the forecast day, the weekday, the day's maximum temperature, and the
#      seasonal-naive's MASE on the same origin. Which of them are hard days for
#      everybody, and which are hard days for the MODEL?
#   2. How many origins did the champion score MASE > 1 on?
#   3. Now the question the table cannot answer: which regimes are NOT in this window?
#      Check df["is_eid"] and df["temp_c"] across 2023-11-01 .. 2023-12-30.
#      Then refit honestly at a cut before Eid al-Fitr (21-24 April 2023) and score
#      the Eid days against the ordinary days that follow them.
#
# HINT: "we have no evidence about X" is a finding, and it belongs in the report next
#       to the table — not omitted because the table looks good without it.

### Mini-quiz

Answer these in your pair before you look at the next cell.

1. Why is a single train/test split not decision-grade evidence?
2. Expanding or sliding — which one forgets old data?
3. What does MASE < 1 mean?
4. Why does MAPE fail near zero?
5. What does a non-significant Diebold-Mariano result tell you to do?

### Task 6 — `BACKTEST.md` and commit

The report is the deliverable, not the notebook. Five short sections, in this order:

1. **The design.** Expanding window, one-year initial window, step 24 h, horizon 24 h, 60
   origins from 2023-11-01 to 2023-12-30, training data ending at the 30 September boundary
   (monthly retrain cadence — say so), MASE denominator = in-sample seasonal-naive over the
   preceding year.
2. **The table**, mean and spread, every model relative to the seasonal-naive baseline.

```
model                      MAE  MAPE_%   MASE   MASE sd   vs baseline
LightGBM (direct)        719.1   2.328  0.445     0.196      -42.2%
LightGBM (quantile q50)  735.4   2.381  0.455     0.203      -40.9%
Seasonal-naive          1244.0   4.117  0.770     0.474           --
```

3. **The DM verdict.** LightGBM vs seasonal-naive: dm = -6.75, p = 0.000, significant.
   LightGBM vs quantile q50: dm = -1.901, p = 0.0575, **not** significant at 5%.
4. **The champion and the rule.** The quantile q50 model, because it is statistically
   indistinguishable from the direct model on accuracy and it also produces the interval the
   reserve decision consumes. State the rule: *among models that are not
   DM-distinguishable, take the one that is simpler, cheaper, or more useful downstream.*
5. **The conditions under which the champion loses**, named as dates and regimes, plus the
   regimes this backtest window contains **no** evidence about.

```
git add -A && git commit -m "feat(eval): 60-origin rolling backtest, DM significance test and champion selection"
```